<a href="https://colab.research.google.com/github/kylashrao/DataScience-Projects/blob/main/Project_(ANN_from_Scratch_with_NumPy_%26_Scikit_learn).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building an Artificial Neural Network (ANN) from scratch using core Python data science libraries allows you to understand the inner workings of deep learning without relying straight away on specialized frameworks like TensorFlow or PyTorch.We will build a multi-layer perceptron (ANN) using NumPy for forward and backward propagation (calculating gradients via the chain rule), Pandas and Scikit-learn for data preprocessing, and SciPy optionally for advanced numerical utilities.Part 1: The Theory & Math of ANNsAn ANN consists of layers of interconnected nodes (neurons). Training involves two main phases:Forward Propagation: Inputs ($X$) are multiplied by weights ($W$), biases ($b$) are added, and an activation function (like Sigmoid or ReLU) is applied to generate a prediction ($\hat{y}$).Backward Propagation (Backprop): The error (Loss) is calculated using Binary Cross-Entropy. Using calculus (chain rule), the gradients of the loss with respect to each weight and bias are computed, and weights are updated iteratively using Gradient Descent:$$W = W - \eta \cdot dW$$(where $\eta$ is the learning rate).Part 2: End-to-End Project (ANN from Scratch with NumPy & Scikit-learn)We will build a simple 2-layer Neural Network (Input $\to$ Hidden Layer $\to$ Output Layer) to solve a binary classification problem using the breast cancer dataset.

1. Import Libraries & Load Data

In [21]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
np.random.seed(42)

# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

2. Feature Scaling
Neural networks require scaled features so that gradient updates remain stable and convergence is smooth.

In [22]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape target for matrix operations: shape (n_samples, 1)
y_train_arr = y_train.reshape(-1, 1)
y_test_arr = y_test.reshape(-1, 1)

3. Implement Activation Functions & Derivatives
We will use the Sigmoid activation function for both the hidden and output layers, along with its derivative for backpropagation.

In [23]:
def sigmoid(z):
  return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # Clip to prevent overflow


def sigmoid_derivative(sig):
  return sig * (1 - sig)

4. Build and Train the Neural Network Class
We encapsulate our ANN parameters, forward propagation, loss calculation, and backpropagation inside a custom Python class.

In [24]:
class ScratchANN:

  def __init__(self, input_dim, hidden_dim, learning_rate=0.01):
    self.lr = learning_rate
    # Initialize weights and biases randomly
    self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
    self.b1 = np.zeros((1, hidden_dim))
    self.W2 = np.random.randn(hidden_dim, 1) * np.sqrt(2.0 / hidden_dim)
    self.b2 = np.zeros((1, 1))

  def forward(self, X):
    self.Z1 = np.dot(X, self.W1) + self.b1
    self.A1 = sigmoid(self.Z1)
    self.Z2 = np.dot(self.A1, self.W2) + self.b2
    self.A2 = sigmoid(self.Z2)
    return self.A2

  def backward(self, X, y, output):
    m = X.shape[0]

    # Binary cross-entropy gradient at the output layer
    dZ2 = output - y
    dW2 = np.dot(self.A1.T, dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # Gradient for hidden layer using chain rule
    dZ1 = np.dot(dZ2, self.W2.T) * sigmoid_derivative(self.A1)
    dW1 = np.dot(X.T, dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    # Update weights and biases via Gradient Descent
    self.W1 -= self.lr * dW1
    self.b1 -= self.lr * db1
    self.W2 -= self.lr * dW2
    self.b2 -= self.lr * db2

  def fit(self, X, y, epochs=1000):
    for epoch in range(epochs):
      output = self.forward(X)
      self.backward(X, y, output)

      if (epoch + 1) % 200 == 0:
        # Binary Cross-Entropy Loss
        loss = -np.mean(
            y * np.log(output + 1e-8) + (1 - y) * np.log(1 - output + 1e-8)
        )
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss:.4f}')

  def predict(self, X):
    probabilities = self.forward(X)
    return (probabilities >= 0.5).astype(int)


# Instantiate and train the model
input_dim = X_train_scaled.shape[1]
ann = ScratchANN(input_dim=input_dim, hidden_dim=8, learning_rate=0.1)

print('--- Training Custom NumPy ANN ---')
ann.fit(X_train_scaled, y_train_arr, epochs=1000)

--- Training Custom NumPy ANN ---
Epoch [200/1000], Loss: 0.1583
Epoch [400/1000], Loss: 0.0987
Epoch [600/1000], Loss: 0.0804
Epoch [800/1000], Loss: 0.0713
Epoch [1000/1000], Loss: 0.0655


5. Model Evaluation
Evaluate how well the scratch-built neural network generalizes to unseen test data.

In [25]:
# Predict on test data
y_pred = ann.predict(X_test_scaled)

# Metrics
print('\n--- Scratch ANN Performance ---')
print(f'Test Accuracy: {accuracy_score(y_test_arr, y_pred):.4f}\n')
print('Confusion Matrix:')
print(confusion_matrix(y_test_arr, y_pred))
print('\nClassification Report:')
print(classification_report(y_test_arr, y_pred, target_names=data.target_names))


--- Scratch ANN Performance ---
Test Accuracy: 0.9912

Confusion Matrix:
[[42  1]
 [ 0 71]]

Classification Report:
              precision    recall  f1-score   support

   malignant       1.00      0.98      0.99        43
      benign       0.99      1.00      0.99        71

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114

